Taller 1

In [ ]:
"""
Funciones que calculan métodos numéricos
"""

import sympy as sp


def Biseccion(funcion,a,b,tolerancia):
    
    if funcion(a) * funcion(b) > 0:
        print('La función no cumple el teorema en el intervalo inicial')
        return
    else:
        iteracion = 0
        while (abs(b-a) > tolerancia):
            iteracion+=1
            p = (a+b)/2
            if (funcion(a)*funcion(p)) > 0:
                a = p
            else:
                b = p
        return iteracion,p

def falsa_posicion(funcion,a,b,tolerancia):
    
    if funcion(a) * funcion(b) > 0:
        print('La función no cumple el teorema en el intervalo inicial')
        return
    else:
        iteracion = 0
        p = b-funcion(b)*(a-b)/(funcion(a)-funcion(b))
        while (abs(funcion(p))>tolerancia):
            iteracion+=1
            p = b-funcion(b)*(a-b)/(funcion(a)-funcion(b))
            if (funcion(a)*funcion(p)) > 0:
                a = p
            else:
                b = p
        return iteracion,p

def newton_rapshon(funcion,x0,tolerancia,x):
    derivada_funcion = sp.diff(funcion,x)
    x1 : float = x0 - (funcion.evalf(subs={x:x0})/derivada_funcion.evalf(subs={x: x0}))
    iteraciones = 0
    while abs(x1 - x0) > tolerancia:
        iteraciones += 1
        x0 = x1
        x1 = x0 - (funcion.evalf(subs={x: x0}) / derivada_funcion.evalf(subs={x: x0}))
    return iteraciones, x1


def secante(funcion, x0, x1, tolerancia):
    iteraciones = 0
    error = 1
    while (error > tolerancia):
        iteraciones += 1
        x2 = x1 - (funcion(x1) * (x0-x1))/(funcion(x0) - funcion(x1))
        error = abs(x2 - x1)
        x0 = x1
        x1 = x2
    
    return iteraciones, x2




Taller 2

In [ ]:
import numpy as np
import time

def gauss_seidel_matrices(A, b, x0, tol):
    """
    Implementación del método de Gauss Seidel.
    -----------
    Parámetros
    -----------
    - A: matriz de coeficientes cuadrada
    - b: vector de términos independientes
    - x0: vector inicial --> Si no me da el dato se asume que el vector es el vector nulo
    - tol: Exactitud con la cual encontraremos la solución del sistema de ecuaciones lineales (SEL). Si no me dan este dato se asume que la tolerancia es 1e-6
    -----------
    Nota
    ------------
    - TANTO LA MATRIZ A COMO EL VECTOR b Y X0 TIENEN QUE SER DE TIPO FLOTANTE
    - AMBAS TIENE QUE TENER DIAGONAL ESTRICTAMENTE DOMINANTE
    -------------
    Retorna:
        x0: solución final
        error_final: error de la última iteración
        errores: lista de errores por iteración
        iteraciones: lista de número de iteración (1, 2, ...)
        time_total: tiempo total de ejecución
    """
    D = np.diag(np.diag(A)) # Obtenemos la matriz diagonal
    L = D - np.tril(A) # Obtenemos la matriz inferior
    U = D - np.triu(A) # Obtenemos la matriz superior
    Tg = np.dot(np.linalg.inv(D-L), U)
    Cg = np.dot(np.linalg.inv(D-L), b)
    v_propios, vect_propios = np.linalg.eig(Tg)
    radio = max(abs(v_propios))
    print(f"Radio espectral: {radio}")
    if radio<1:
        import time
        time_start = time.time()
        error = 1
        iteracion = 1
        errores = []
        while (error > tol):
            x1 = np.dot(Tg,x0) + Cg
            error = np.max(np.abs(x1-x0))
            errores.append(error)
            x0 = np.copy(x1)
            iteracion += 1
        time_end = time.time()
        time_total = time_end - time_start
        return x0, error, errores, time_total, iteracion-1
            
    else:
        print("El sistema iterativo no converge con el método de Jacobi")
        return None, None, [], 0, 0

import numpy as np

def gauss_seidel_sumas(A, b, x0=None, tol=1e-10, max_iter=100):
    """
    Método de Gauss-Seidel para resolver Ax = b con suma de cambios por iteración.
    
    Parámetros:
        A : array-like (n x n)
        b : array-like (n)
        x0 : array-like (n), condiciones iniciales (opcional)
        tol : tolerancia para el criterio de parada
        max_iter : número máximo de iteraciones
    
    Retorna:
        x : solución aproximada (array de NumPy)
        error_final : error de la última iteración
        errores : lista con el error por iteración (norma infinito)
        iteraciones : lista de número de iteración (1, 2, ...)
        tiempo_total : tiempo de ejecución
    """
    import time
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    if x0 is None:
        x = np.zeros_like(b, dtype=float)
    else:
        x = np.array(x0, dtype=float)

    suma_iteraciones = []
    errores = []

    time_start = time.time()

    for it in range(1, max_iter+1):
        x_new = x.copy()
        suma_cambios = 0.0

        for i in range(n):
            suma = np.dot(A[i, :i], x_new[:i]) + np.dot(A[i, i+1:], x[i+1:])
            nuevo_valor = (b[i] - suma) / A[i, i]
            suma_cambios += abs(nuevo_valor - x[i])
            x_new[i] = nuevo_valor

        suma_iteraciones.append(suma_cambios)
        error = np.max(np.abs(x_new - x))
        errores.append(error)

        if suma_cambios < tol:
            break

        x = x_new

    time_end = time.time()
    tiempo_total = time_end - time_start
    error_final = errores[-1] if errores else None
    iteraciones = it

    return x, error_final, errores, tiempo_total, iteraciones


def jacobi_con_matrices(A,b,x0,tol):
    """
    Implementación del método de Jacobi utilizando matrices.
    -----------
    Parámetros
    -----------
    - A: matriz de coeficientes cuadrada
    - b: vector de términos independientes
    - x0: vector inicial --> Si no me da el dato se asume que el vector es el vector nulo
    - tol: Exactitud con la cual encontraremos la solución del sistema de ecuaciones lineales (SEL). Si no me dan este dato se asume que la tolerancia es 1e-6
    -----------
    Nota
    ------------
    - TANTO LA MATRIZ A COMO EL VECTOR b Y X0 TIENEN QUE SER DE TIPO FLOTANTE
    - AMBAS TIENE QUE TENER DIAGONAL ESTRICTAMENTE DOMINANTE
    """
    D = np.diag(np.diag(A)) # Obtenemos la matriz diagonal
    L = D - np.tril(A) # Obtenemos la matriz inferior
    U = D - np.triu(A) # Obtenemos la matriz superior
    Tj = np.dot(np.linalg.inv(D), L+U)
    Cj = np.dot(np.linalg.inv(D), b)
    v_propios, vect_propios = np.linalg.eig(Tj)
    radio = max(abs(v_propios))
    print(radio)
    if radio<1:
        time_start = time.time()
        error = 1
        iteracion = 1
        errores = []
        while (error > tol):
            x1 = np.dot(Tj,x0) + Cj
            error = np.max(np.abs(x1-x0))
            errores.append(error)
            x0 = np.copy(x1)
            # print(f"Iteración {iteracion}: {x1}, Error: {error}")
            iteracion += 1
        time_end = time.time()
        time_total = time_end - time_start
        return x0, error, errores, time_total, iteracion
            
    else:
        print("El sistema iterativo no converge con el método de Jacobi")

def jacobi_con_sumas(A, b, x0, n_max, tol):
    """
    -----------
    Parametros
    -----------
    - A: matriz de coeficientes cuadrada
    - b: vector de términos independientes
    - x0: vector inicial --> Si no me da el dato se asume que el vector es el vector nulo
    - n_max: número máximo de iteraciones
    - tol: Exactitud con la cual encontraremos la solución del sistema de ecuaciones lineales (SEL). Si no me dan este dato se asume que la tolerancia es 1e-6
    -----------
    Nota
    ------------
    - TANTO LA MATRIZ A COMO EL VECTOR b Y X0 TIENEN QUE SER DE TIPO FLOTANTE
    - AMBAS TIENE QUE TENER DIAGONAL ESTRICTAMENTE DOMINANTE
    """

    n = len(b)  # -> Tamaño del vector b que es el mismo de A porque es cuadrada
    x1 = np.zeros(n)
    error = 10
    iteracion = 0
    errores = []
    
    
    time_start = time.time()
    while error > tol and iteracion < n_max:
        for i in range(n):
            suma = 0  # Inicializar suma para cada fila
            for j in range(n):
                if i != j:
                    suma += np.dot(A[i][j], x0[j])  # Acumular la suma correctamente
            x1[i] = (b[i] - suma) / A[i][i]
        
        # Calculo el error como la norma infinito de la diferencia
        error = np.max(np.abs(x1 - x0))
        errores.append(error)
        
        # Actualizo el vector inicial
        x0 = np.copy(x1)
        iteracion += 1
        
        # print(f"Iteración {iteracion}: {x1}, Error: {error}")
    time_end = time.time()
    tiempo_total = time_end - time_start
    
    return x1, error, errores, tiempo_total, iteracion

import numpy as np

def minimos_cuadrados(x, y):
    """
    Ajuste de mínimos cuadrados para una función lineal.
    
    Parameters:
    x (array-like): Valores de la variable independiente.
    y (array-like): Valores de la variable dependiente.
    
    Returns:
    tuple: Coeficientes del polinomio ajustado (a0, a1).
    """
    Sx = np.sum(x)
    Sy = np.sum(y)
    Sx2 = np.sum(x**2)
    Sxy = np.sum(x*y)
    n = len(x) # numero de datoso

    a0 = ((Sy * Sx2) - (Sx * Sxy)) / ((n * Sx2) - (Sx)**2)
    a1 = ((n * Sxy) - (Sx * Sy)) / ((n * Sx2) - (Sx)**2)

    return a0, a1

import numpy as np

def metodo_euler_profe(f,a,b,h,y0):
    """
    Parameters
    ----------
    f : function
        Ecuación diferencial a aproximar.
    a : float
        Valor del intervalo en a.
    b : float
        Valor del intervalo en b.
    h : float
        Esparcimiento.
    y0 : list
        Condiciones iniciales de la ecuación diferencial.
    Returns
    -------
    tiempos : numpy.ndarray
        Arreglo de tiempos discretizados.
    weu : list
        Aproximación de la ecuación diferencial por el método de Euler.
    """
    weu = [y0]
    n = int((b-a)/h)
    for i in range(n):
        weu.append(weu[i] + h*f(a + i*h,weu[i]))
    return np.linspace(a,b,n+1),weu

def runge_kutta(f,a,b,h,y0):
    n = int((b-a)/h)
    wrk=[y0]
    for i in range(n):
        k1= h*f(a+i*h, wrk[i])
        k2= h*f(a+i*h+0.5*h, wrk[i]+0.5*k1)
        k3= h*f(a+i*h+0.5*h, wrk[i]+0.5*k2)
        k4= h*f(a+(i+1)*h, wrk[i]+k3)
        wrk.append(wrk[i]+(1/6)*(k1+2*k2+2*k3+k4))
    return np.linspace(a,b,n+1),wrk

import sympy as sp

def lagrange_pol(x_data, y_data):
    x = sp.symbols('x')
    
    P = 0 # Acumulador del polinomio
    n = len(x_data) # Número de datos
    
    for i in range(n):
        Li = 1
        for j in range(n):
            if j != i:
                Li *= (x - x_data[j]) / (x_data[i] - x_data[j])
        P += Li * y_data[i]
    return sp.expand(P)

import numpy as np
import matplotlib.pyplot as plt


def graficar_escalas(x_d, y_d):
    """
    Función que grafica los datos en escalas diferentes escalas.

    Parámetros
    ----------
    x_d : array
        Datos en el eje x.
    y_d : array
        Datos en el eje y.
    Returns
    -------
    graficas : list
        Lista de graficas generadas.
        
    """
    plt.figure(figsize=(12, 12), dpi=100)
    plt.subplot(331)
    plt.plot(x_d , y_d, 'or', label='Datos observados') # graficos de dispersión
    plt.subplot(332)
    plt.plot(x_d, np.sqrt(y_d), 'ob')
    plt.xlabel('x')
    plt.ylabel('$\sqrt{y}$')
    plt.subplot(333)
    plt.plot(x_d, 1/y_d, 'ob')
    plt.xlabel('x')
    plt.ylabel('$1/y$')
    plt.subplot(334)
    plt.plot(x_d**2, y_d, 'ob')
    plt.xlabel('$x^2$')
    plt.ylabel('y')
    plt.subplot(335)
    plt.plot(x_d**3, y_d, 'ob')
    plt.xlabel('$x^3$')
    plt.ylabel('y')
    plt.subplot(336)
    plt.plot(np.log(x_d), y_d, 'ob')
    plt.xlabel('$\log(x)$')
    plt.ylabel('y')
    plt.subplot(337)
    plt.plot(x_d, np.log(y_d), 'ob')
    plt.ylabel('$\log(y)$')
    plt.xlabel('x')
    plt.subplot(338)
    plt.plot(np.sqrt(x_d), y_d, 'ob')
    plt.xlabel('$\sqrt{x}$')
    plt.ylabel('y')
    plt.subplot(339)
    plt.plot(np.log(x_d), np.log(y_d), 'ob')
    plt.xlabel('$\log(x)$')
    plt.ylabel('$\log(y)$')
    plt.subplots_adjust(wspace=0.4, hspace=0.6)  # Ajusta el espacio entre subgráficos
    return plt